# Task B – MFCD vs VCD (Self-contained)
This notebook clones the repo, installs dependencies, downloads COCO val2014, runs VCD and MFCD, then prints a comparison table.

In [37]:
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

Torch: 2.2.2+cu121
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:218: UserWarning: 
NVIDIA RTX PRO 6000 Blackwell Server Edition with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA RTX PRO 6000 Blackwell Server Edition GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [19]:
import os
ROOT = '/content/VCD_project'
REPO_URL = 'https://github.com/maxheef/ECE209_Project1.git'
BRANCH = 'main'
COCO_VAL = '/content/datasets/coco/val2014'
MODEL_PATH = 'liuhaotian/llava-v1.5-7b'
SEED = 55

# MFCD parameters (from MFCD readme)
MFC_HIGH_ALPHA = 1.0
MFC_LOW_ALPHA = 1.0
MFC_BETA = 0.3
MFC_HIGH_PASS_CUTOFF = 0.1
MFC_LOW_PASS_CUTOFF = 0.9
MFC_FILTER_TYPE = 'gaussian'
print('ROOT=', ROOT)

ROOT= /content/VCD_project


In [38]:
%%bash
set -euo pipefail
cd /content
if [ -d /content/VCD_project/.git ]; then
  git -C /content/VCD_project fetch --depth 1 origin ${BRANCH:-main}
  git -C /content/VCD_project reset --hard FETCH_HEAD
else
  rm -rf /content/VCD_project
  git clone --depth 1 --branch ${BRANCH:-main} ${REPO_URL:-https://github.com/maxheef/ECE209_Project1.git} /content/VCD_project
fi

if [ ! -d /content/VCD_project/originalMFCD/mfcd ]; then
  echo 'Missing /content/VCD_project/originalMFCD/mfcd (check repo contents)'.
  find /content/VCD_project -maxdepth 3 -type d | sed -n '1,120p'
  exit 1
fi


HEAD is now at 96386fb requirements.txt edited


From https://github.com/maxheef/ECE209_Project1
 * branch            main       -> FETCH_HEAD
 + 18d685c...96386fb main       -> origin/main  (forced update)


In [39]:
%%bash
set -euo pipefail
REQ=$(find /content/VCD_project/originalMFCD -maxdepth 3 -name requirements.txt | head -n 1)
if [ -z "$REQ" ]; then
  echo 'Could not find MFCD requirements.txt under /content/VCD_project/originalMFCD'
  find /content/VCD_project/originalMFCD -maxdepth 3 -type d | sed -n '1,120p'
  exit 1
fi
echo "Using requirements: $REQ"
pip install -r "$REQ"
# pip install --upgrade --index-url https://download.pytorch.org/whl/cu121 torch==2.2.2 torchvision==0.17.2
# pip install --force-reinstall 'numpy<2'


Using requirements: /content/VCD_project/originalMFCD/mfcd/requirements.txt


ERROR: Invalid requirement: 'torch>=2.7.0+cu128': Local version label can only be used with `==` or `!=` operators
    torch>=2.7.0+cu128
         ~~~~~~~^ (from line 2 of /content/VCD_project/originalMFCD/mfcd/requirements.txt)


CalledProcessError: Command 'b'set -euo pipefail\nREQ=$(find /content/VCD_project/originalMFCD -maxdepth 3 -name requirements.txt | head -n 1)\nif [ -z "$REQ" ]; then\n  echo \'Could not find MFCD requirements.txt under /content/VCD_project/originalMFCD\'\n  find /content/VCD_project/originalMFCD -maxdepth 3 -type d | sed -n \'1,120p\'\n  exit 1\nfi\necho "Using requirements: $REQ"\npip install -r "$REQ"\n# pip install --upgrade --index-url https://download.pytorch.org/whl/cu121 torch==2.2.2 torchvision==0.17.2\n# pip install --force-reinstall \'numpy<2\'\n'' returned non-zero exit status 1.

In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/datasets/coco
cd /content/datasets/coco
if [ ! -d val2014 ]; then
  wget -q http://images.cocodataset.org/zips/val2014.zip
  unzip -q val2014.zip
fi
ls -la /content/datasets/coco/val2014 | sed -n '1,8p'


In [ ]:
%%bash
set -euo pipefail
# Run VCD baseline (Task A code)
PYTHON_BIN=python3 bash /content/VCD_project/myTasks/run_table1.sh \
  liuhaotian/llava-v1.5-7b \
  /content/datasets/coco/val2014 \
  55


In [ ]:
import os
import json
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset

MFCD = '/content/VCD_project/originalMFCD/mfcd'
ORIG = '/content/VCD_project/originalProject'
P1 = '/content/VCD_project/myTasks'

os.environ['PYTHONPATH'] = MFCD + ':' + os.environ.get('PYTHONPATH','')

from eval_utils import prepare_for_generate, generate_responses, batch_prepare_data
from eval_utils import MODEL_INFOS
from eval.pope.eval import eval as pope_eval, recorder

class POPELocal(Dataset):
    def __init__(self, json_path, image_root):
        self.rows = [json.loads(line) for line in open(json_path, 'r')]
        self.image_root = Path(image_root)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        img = Image.open(self.image_root / row['image']).convert('RGB')
        return {
            'question': row['text'],
            'image': img,
            'label': row['label'].lower(),
        }

def run_mfcd(pope_type):
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    class Args: pass
    args = Args()
    args.model_name_or_path = MODEL_PATH
    args.model_type = 'llava-1.5'
    args.num_workers = 1
    args.eval_batch_size = 1
    args.device = 'cuda:0'
    args.temperature = 1.2
    args.top_p = 1.0
    args.top_k = 50
    args.max_new_tokens = 2048
    args.mfc_high_alpha = MFC_HIGH_ALPHA
    args.mfc_low_alpha = MFC_LOW_ALPHA
    args.mfc_beta = MFC_BETA
    args.mfc_jsd = False
    args.mfc_entropy = False
    args.mfc_high_pass_cutoff = MFC_HIGH_PASS_CUTOFF
    args.mfc_low_pass_cutoff = MFC_LOW_PASS_CUTOFF
    args.mfc_filter_type = MFC_FILTER_TYPE

    processor, generation_config, _ = prepare_for_generate(args=args, model_info=MODEL_INFOS['llava-1.5'], device=device)

    pope_json = f"{ORIG}/experiments/data/POPE/coco/coco_pope_{pope_type}.json"
    dataset = POPELocal(pope_json, COCO_VAL)

    prompts = batch_prepare_data(args=args, dataset=dataset, processor=processor, question_column_name='question', image_column_name='image')

    model_info = MODEL_INFOS['llava-1.5']
    model = model_info.model_class.from_pretrained(MODEL_PATH, **model_info.model_config).to(device)

    responses = generate_responses(model=model, processor=processor, prompts=prompts, generation_config=generation_config)

    labels = [1 if d['label'] == 'yes' else 0 for d in dataset]
    preds = recorder(responses)
    return pope_eval(preds, labels)

mfcd_random = run_mfcd('random')
mfcd_popular = run_mfcd('popular')
print('MFCD random:', mfcd_random)
print('MFCD popular:', mfcd_popular)


In [ ]:
# Merge and print table: VCD (from myTasks output files) vs MFCD (computed above)
import re
from pathlib import Path

out_dir = Path(f'{P1}/output')
def parse_metrics(path: Path):
    text = path.read_text()
    def g(name):
        m = re.search(rf"^{name}:\s*([0-9.]+)", text, re.MULTILINE)
        return float(m.group(1)) if m else float('nan')
    return {
        'accuracy': g('Accuracy'),
        'precision': g('Precision'),
        'recall': g('Recall'),
        'f1': g('F1'),
    }

vcd_random = parse_metrics(out_dir / f'metrics_random_vcd_seed{SEED}.txt')
vcd_popular = parse_metrics(out_dir / f'metrics_popular_vcd_seed{SEED}.txt')

print('| Split | Method | Accuracy | Precision | Recall | F1 |')
print('|---|---|---:|---:|---:|---:|')
for split, vcd, mfcd in [
    ('random', vcd_random, mfcd_random),
    ('popular', vcd_popular, mfcd_popular),
]:
    print(f"| {split} | VCD | {vcd['accuracy']:.4f} | {vcd['precision']:.4f} | {vcd['recall']:.4f} | {vcd['f1']:.4f} |")
    print(f"| {split} | MFCD | {mfcd['accuracy']:.4f} | {mfcd['precision']:.4f} | {mfcd['recall']:.4f} | {mfcd['f1_score']:.4f} |")
